In [0]:
%run "./00_config"

In [0]:
df = (
    spark.read
         .option("wholetext", True)
         .text(source_path + "*.json")
         .select(
              lit(source_path).alias("file_path"),
             regexp_extract(col("_metadata.file_path"), r"([^/]+)$", 1).alias("file_name"),
             col("value").alias("json_original_data"),
             current_timestamp().alias("loaded_at")
         )
)



In [0]:
df_final = df.select(
    "file_path",
    "file_name",
    "json_original_data", # Original JSON
    parse_json("json_original_data").alias("json_data_for_parsing"),  # Queryable JSON, VARIANT dtype
    "loaded_at"
)

In [0]:
df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{tgt_catalog}.{tgt_schema}.{tgt_table}")

In [0]:
%sql
SELECT * FROM iics.information_schema.columns

In [0]:
df = spark.read.table("iics.casemix.json_raw_test1")

In [0]:
df.dtypes

In [0]:
%sql
SELECT * FROM iics.casemix.json_raw_test1

In [0]:
%sql
SELECT * FROM iics.casemix.json_test

In [0]:
from pyspark.sql.functions import (
    col,
    to_json,
    parse_json,
    current_timestamp,
    input_file_name
)


In [0]:
 
df_option_1_variant = (spark.read.text("/Volumes/iics/casemix/history/json_test/extracted/carereport_hsar_20250305_162803/carereport_hsar_5110097798747137794_20250224_20250305_20250305_162803.json")
        .withColumn("payload", parse_json(col("value")))
        .withColumn("ingestion_time", current_timestamp()))
 


In [0]:
df_option_1_variant.write.format("delta").mode("append").saveAsTable("iics.casemix.json_option_1_events_variant_test")

In [0]:
%sql
SELECT
    payload:id::STRING AS report_id,
    payload:resourceType::STRING AS resource_type,
    payload:title::STRING AS title,
    payload:status::STRING AS status,
    payload:date::STRING AS report_date,
    payload:author[0].display::STRING AS author_name,
    payload:subject.display::STRING AS patient_name,
    ingestion_time
FROM iics.casemix.json_option_1_events_variant_test;


In [0]:
df_option_2 = (
    spark.read
         .option("multiLine", True)
         .json("/Volumes/iics/casemix/history/json_test/extracted/carereport_hsar_20250305_162803/carereport_hsar_5110097798747137794_20250224_20250305_20250305_162803.json")
)

In [0]:
df_option_2 = (
    df_option_2.select(
        to_json(col("author")).alias("author"),
        col("confidentiality"),
        to_json(col("contained")).alias("contained"),
        to_json(col("custodian")).alias("custodian"),
        col("date"),
        to_json(col("extension")).alias("extension"),
        col("id"),
        to_json(col("meta")).alias("meta"),
        col("resourceType"),
        to_json(col("section")).alias("section"),
        col("status"),
        to_json(col("subject")).alias("subject"),
        col("title"),
        to_json(col("type")).alias("type"),
        current_timestamp().alias("loaded_at")
    )
)



In [0]:
df_option_2.write.mode("append").saveAsTable("iics.casemix.json_option_2_events_variant_test")


### Option 1 - Varaint Column

In [0]:
df= spark.read.table("iics.casemix.json_option_1_events_variant_test")

In [0]:
df.dtypes

In [0]:
%sql
describe iics.casemix.json_option_1_events_variant_test

In [0]:
%sql
describe iics.casemix.json_option_2_events_variant_test

In [0]:
%sql
SELECT * FROM iics.casemix.json_option_1_events_variant_test

%md
### Option 2 - parsed json

In [0]:
%sql
SELECT * FROM iics.casemix.json_option_2_events_variant_test

In [0]:
%sql
DESCRIBE iics.casemix.json_test

In [0]:
%sql
select * from iics.casemix.json_test